# T-3 Forecasting Benchmark Report


In [7]:
from cybench.evaluation.eval import implemented_metrics
from cybench.util.config_utils import get_run_description
import pathlib
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import os
from cybench.config import (
    DATASETS,
    PATH_DATA_DIR,
    PATH_RESULTS_DIR,
    KEY_COUNTRY,
    KEY_LOC,
    KEY_YEAR,
    KEY_TARGET,
)

def calculate_metrics_for_run(df: pd.DataFrame, run_name: str):
    """Calculates all defined metrics for a single run DataFrame."""
    if 'targets' not in df.columns or 'preds' not in df.columns:
        return {}

    y_true = df['targets'].values
    y_pred = df['preds'].values
    mask = (~np.isnan(y_true)) & (~np.isnan(y_pred))
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    results = {}
    for name, func in implemented_metrics.items():
        try:
            results[name] = np.round(func(y_true, y_pred), 4)
        except Exception:
            results[name] = np.nan
    return results

# --- Main Analysis Function ---

def analyze_experiments(exp_path: str):
    """Loads all runs, calculates metrics, and outputs a styled table."""
    exp_dir = pathlib.Path(exp_path)
    if not exp_dir.exists():
        return f"Error: Experiment path not found: {exp_path}"

    all_results = []

    for run_dir in exp_dir.iterdir():
        if run_dir.is_dir() and run_dir.name != '.hydra':

            # Find preds.csv recursively to handle nested structures
            preds_file = next(run_dir.glob('**/preds.csv'), None)
            overrides_path = run_dir / '.hydra' / 'overrides.yaml'

            if preds_file and overrides_path.exists():
                run_description = get_run_description(overrides_path)

                try:
                    df_run = pd.read_csv(preds_file)
                    metrics_dict = calculate_metrics_for_run(df_run, run_description)

                    if metrics_dict:
                        metrics_dict['Run Description'] = run_dir #run_description
                        all_results.append(metrics_dict)
                except Exception as e:
                    print(f"Error processing {run_dir.name}: {e}")

    if not all_results:
        return "No successful runs were processed or data files were found."

    # Create the final results DataFrame
    final_df = pd.DataFrame(all_results).set_index('Run Description')

    # Ensure 'r2' is the first column if it exists
    if 'r2' in final_df.columns:
        cols = ['r2'] + [col for col in final_df.columns if col != 'r2']
        final_df = final_df[cols]
    return final_df
    # Style the DataFrame: RdBu colormap from -1 to 1 for R2
    styled_df = final_df.style.background_gradient(
        subset=['r2'],
        cmap='RdBu',
        vmin=-1,
        vmax=1
    ).format(
        precision=4 # Format all numeric columns
    ).set_caption("Experiment Evaluation Summary")

    return styled_df

In [8]:
exp_path = "../cybench/output/playy"

analyze_experiments(exp_path)

,r2,mse,normalized_rmse,mape,r
Run Description,,,,,
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_104005,0.2992,2.3503,26.8709,7.368173e+13,0.6383
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_110718,0.3641,2.1326,25.5957,6.970788e+13,0.6597
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_111200,0.2437,2.5365,27.9147,7.962026e+13,0.6525
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_111401,0.2078,2.6567,28.5684,8.446328e+13,0.6231
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_111526,0.3800,2.0793,25.2744,7.665538e+13,0.6421
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_111728,0.3131,2.3037,26.6030,7.288668e+13,0.6272
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_111852,0.3228,2.2712,26.4144,7.458309e+13,0.6464
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_112039,0.1903,2.7154,28.8827,7.876478e+13,0.5312
..\cybench\output\playy\maize_AR_cnn_lf_single_20251209_112207,0.3966,2.0237,24.9341,7.396511e+13,0.6884
